# Visualización de Activaciones del Modelo Facial

Este notebook permite visualizar cómo el modelo CNN "ve" las imágenes en sus diferentes capas.
Las visualizaciones se muestran directamente en el notebook sin guardar archivos.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow import keras
import random

# Configurar matplotlib para mostrar imágenes inline
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

# Configurar paths para importar módulos del proyecto
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Importar módulos del proyecto
try:
    from models.facial.classifiers.cnn_classifier import load_cnn_classifier, IMAGE_SIZE
    print("✓ Módulos importados correctamente")
except ImportError as e:
    print(f"✗ Error importando módulos: {e}")
    print(f"Sys path: {sys.path}")

In [ ]:
# Cargar el modelo
try:
    model, label_encoder = load_cnn_classifier()
    print("✓ Modelo cargado exitosamente")
    print(f"\nClases detectadas: {label_encoder.classes_}")
    print(f"Número de capas: {len(model.layers)}")
    model.summary()
except Exception as e:
    print(f"✗ Error cargando el modelo: {e}")

## Funciones de Visualización

In [ ]:
def predict_and_show_image(image_path):
    """
    Muestra la imagen y su predicción
    """
    image = Image.open(image_path).convert("RGB")
    if image.size != (IMAGE_SIZE, IMAGE_SIZE):
        image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
    
    img_array = np.array(image, dtype=np.float32) / 255.0
    img_array_batch = np.expand_dims(img_array, axis=0)
    
    # Hacer predicción
    prediction = model.predict(img_array_batch, verbose=0)
    predicted_class_idx = np.argmax(prediction[0])
    predicted_class = label_encoder.classes_[predicted_class_idx]
    confidence = prediction[0][predicted_class_idx] * 100
    
    # Mostrar imagen con predicción
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.title(f"Imagen: {Path(image_path).parent.name}/{Path(image_path).name}\n"
              f"Predicción: {predicted_class} ({confidence:.1f}%)",
              fontsize=12, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    
    return img_array_batch, predicted_class, confidence

In [ ]:
def visualize_layer_activations(image_path, max_filters=16):
    """
    Visualiza las activaciones de cada capa del modelo CNN
    """
    print(f"\n{'='*80}")
    print(f"VISUALIZACIÓN DETALLADA: {Path(image_path).name}")
    print(f"{'='*80}\n")
    
    # Mostrar imagen y predicción
    img_array, predicted_class, confidence = predict_and_show_image(image_path)
    
    # Obtener las capas de interés
    layer_outputs = []
    layer_names = []
    
    for layer in model.layers:
        if isinstance(layer, (keras.layers.Conv2D, keras.layers.MaxPooling2D, 
                            keras.layers.Activation, keras.layers.Add)):
            layer_outputs.append(layer.output)
            layer_names.append(f"{layer.name} ({layer.__class__.__name__})")
    
    # Crear modelo de activación
    activation_model = keras.models.Model(inputs=model.input, outputs=layer_outputs)
    activations = activation_model.predict(img_array, verbose=0)
    
    # Visualizar activaciones
    for layer_idx, (layer_name, activation) in enumerate(zip(layer_names, activations)):
        if len(activation.shape) == 4:
            n_features = activation.shape[-1]
            n_cols = min(8, n_features)
            n_rows = min(max_filters // n_cols, (n_features + n_cols - 1) // n_cols)
            n_to_show = min(max_filters, n_features)
            
            fig = plt.figure(figsize=(n_cols * 2, n_rows * 2))
            fig.suptitle(f"Capa {layer_idx + 1}: {layer_name}\nShape: {activation.shape[1:]}", 
                       fontsize=12, fontweight="bold")
            
            for i in range(n_to_show):
                ax = plt.subplot(n_rows, n_cols, i + 1)
                feature_map = activation[0, :, :, i]
                feature_map = (feature_map - feature_map.min()) / (feature_map.max() - feature_map.min() + 1e-8)
                ax.imshow(feature_map, cmap="viridis")
                ax.set_title(f"Filtro {i + 1}", fontsize=8)
                ax.axis("off")
            
            plt.tight_layout()
            plt.show()

In [ ]:
def visualize_specific_blocks(image_path):
    """
    Visualiza solo los bloques principales (después de MaxPooling)
    """
    print(f"\n{'='*80}")
    print(f"VISUALIZACIÓN POR BLOQUES: {Path(image_path).name}")
    print(f"{'='*80}\n")
    
    # Mostrar imagen y predicción
    img_array, predicted_class, confidence = predict_and_show_image(image_path)
    
    # Obtener capas de MaxPooling
    block_layers = []
    block_names = []
    for i, layer in enumerate(model.layers):
        if isinstance(layer, keras.layers.MaxPooling2D):
            block_layers.append(layer.output)
            block_names.append(f"Bloque {len(block_layers)} (MaxPooling)")
    
    activation_model = keras.models.Model(inputs=model.input, outputs=block_layers)
    activations = activation_model.predict(img_array, verbose=0)
    
    # Visualizar cada bloque
    for block_idx, (block_name, activation) in enumerate(zip(block_names, activations)):
        n_features = activation.shape[-1]
        n_to_show = min(16, n_features)
        n_cols = 8
        n_rows = (n_to_show + n_cols - 1) // n_cols
        
        fig = plt.figure(figsize=(16, n_rows * 2))
        fig.suptitle(f"{block_name} - Shape: {activation.shape[1:]}", 
                    fontsize=14, fontweight="bold")
        
        for i in range(n_to_show):
            ax = plt.subplot(n_rows, n_cols, i + 1)
            feature_map = activation[0, :, :, i]
            feature_map = (feature_map - feature_map.min()) / (feature_map.max() - feature_map.min() + 1e-8)
            ax.imshow(feature_map, cmap="viridis")
            ax.set_title(f"Filtro {i + 1}", fontsize=9)
            ax.axis("off")
        
        plt.tight_layout()
        plt.show()

## Preparar Ejemplos de Imágenes

Buscar imágenes de ejemplo de diferentes categorías.

In [ ]:
# Buscar imágenes organizadas por categoría
data_dir = project_root / "models" / "facial" / "data"

# Organizar imágenes por categoría
images_by_category = {}
for category_dir in data_dir.iterdir():
    if category_dir.is_dir():
        images = list(category_dir.glob("*.jpg")) + list(category_dir.glob("*.png"))
        if images:
            images_by_category[category_dir.name] = images

print(f"Categorías encontradas: {list(images_by_category.keys())}")
print(f"\nTotal de imágenes por categoría:")
for category, images in images_by_category.items():
    print(f"  - {category}: {len(images)} imágenes")

# Seleccionar imágenes de ejemplo (2-3 por categoría)
example_images = []
for category, images in images_by_category.items():
    # Tomar 2 imágenes aleatorias de cada categoría
    sample_size = min(2, len(images))
    example_images.extend(random.sample(images, sample_size))

print(f"\n✓ Se seleccionaron {len(example_images)} imágenes de ejemplo")

## Ejemplo 1: Visualización Rápida por Bloques

Visualiza las activaciones de los bloques principales para múltiples imágenes.

In [ ]:
# Visualizar bloques para las primeras 3 imágenes
num_examples = min(3, len(example_images))

print(f"\n{'#'*80}")
print(f"VISUALIZACIÓN POR BLOQUES - {num_examples} EJEMPLOS")
print(f"{'#'*80}")

for i in range(num_examples):
    visualize_specific_blocks(example_images[i])

## Ejemplo 2: Visualización Detallada de Todas las Capas

Visualiza todas las capas convolucionales para una imagen específica.

In [ ]:
# Visualización detallada de 1 imagen
if example_images:
    print(f"\n{'#'*80}")
    print(f"VISUALIZACIÓN DETALLADA DE TODAS LAS CAPAS")
    print(f"{'#'*80}")
    
    visualize_layer_activations(example_images[0], max_filters=16)

## Ejemplo 3: Comparación entre Categorías

Visualiza cómo el modelo procesa imágenes de diferentes categorías.

In [ ]:
# Seleccionar una imagen de cada categoría
comparison_images = []
for category, images in images_by_category.items():
    comparison_images.append(random.choice(images))

print(f"\n{'#'*80}")
print(f"COMPARACIÓN ENTRE CATEGORÍAS - {len(comparison_images)} IMÁGENES")
print(f"{'#'*80}")

for img_path in comparison_images:
    visualize_specific_blocks(img_path)

## Ejemplo 4: Visualización Personalizada

Puedes especificar manualmente la ruta de una imagen para visualizar.

In [ ]:
# Descomentar y modificar la ruta para visualizar una imagen específica
# custom_image_path = data_dir / "categoria" / "imagen.jpg"
# visualize_specific_blocks(custom_image_path)
# visualize_layer_activations(custom_image_path, max_filters=8)

print("💡 Tip: Descomenta las líneas anteriores y especifica la ruta de tu imagen")

## Resumen

Este notebook te permite:
- ✓ Visualizar activaciones de múltiples imágenes
- ✓ Ver predicciones del modelo con confianza
- ✓ Comparar cómo el modelo procesa diferentes categorías
- ✓ Todas las visualizaciones se muestran inline (sin guardar archivos)
- ✓ Explorar capas específicas o bloques completos

**Nota:** Las imágenes se muestran directamente en el notebook y no se guardan en el almacenamiento.